# Serverless ML Deployment

## What is Serverless?
Serverless means no server management the cloud provider handles infrastructure, scaling, and availability. You pay per request, not per idle server.

## Serverless Options for ML
| Platform | Type | ML Support | Cold Start |
|----------|------|-----------|------------|
| **AWS Lambda** | FaaS | Up to 10GB RAM | 100ms-5s |
| **AWS SageMaker** | Managed ML | Full ML lifecycle | Varies |
| **GCP Cloud Run** | Container | Any containerized model | ~1s |
| **Google Cloud Functions** | FaaS | Lightweight models | <1s |
| **Azure Functions** | FaaS | Moderate models | ~1s |
| **Modal** | ML-native | GPU support | Fast |
| **Fly.io** | Container | Any | Fast |

## Cold Start Problem
When a function hasn't been invoked recently, the container needs to initialize causing latency spikes.

**Solutions:**
- Provisioned concurrency (AWS Lambda)
- Keep-alive pings
- Smaller model/container sizes
- Lazy loading inside handler

In [1]:
# AWS Lambda handler for ML inference
LAMBDA_HANDLER = '''
import json
import numpy as np
import os
import boto3
import joblib
import io

# Load model at cold start (outside handler = cached across invocations)
MODEL = None
S3_BUCKET = os.environ.get('MODEL_BUCKET', 'my-ml-models')
MODEL_KEY = os.environ.get('MODEL_KEY', 'models/iris_classifier.joblib')

def get_model():
    global MODEL
    if MODEL is None:
        print("Cold start: loading model from S3...")
        s3 = boto3.client('s3')
        response = s3.get_object(Bucket=S3_BUCKET, Key=MODEL_KEY)
        model_bytes = response['Body'].read()
        MODEL = joblib.load(io.BytesIO(model_bytes))
        print("Model loaded")
    return MODEL

def lambda_handler(event, context):
    try:
        # Parse request
        if isinstance(event.get('body'), str):
            body = json.loads(event['body'])
        else:
            body = event
        
        features = body['features']
        if not isinstance(features[0], list):
            features = [features]  # single sample
        
        # Predict
        model = get_model()
        X = np.array(features, dtype=np.float32)
        predictions = model.predict(X).tolist()
        probabilities = model.predict_proba(X).tolist()
        
        return {
            'statusCode': 200,
            'headers': {'Content-Type': 'application/json', 'Access-Control-Allow-Origin': '*'},
            'body': json.dumps({
                'predictions': predictions,
                'probabilities': probabilities
            })
        }
    except Exception as e:
        return {
            'statusCode': 500,
            'body': json.dumps({'error': str(e)})
        }
'''

with open('/tmp/lambda_handler.py', 'w') as f:
    f.write(LAMBDA_HANDLER)
print('Lambda handler saved')

Lambda handler saved


In [2]:
# Lambda deployment with container image (for larger models)
DOCKERFILE_LAMBDA = '''
FROM public.ecr.aws/lambda/python:3.11

# Install dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy model and handler
COPY model.joblib ./model.joblib
COPY lambda_handler.py ./lambda_handler.py

# Lambda entry point
CMD ["lambda_handler.lambda_handler"]
'''

# Build and push to ECR:
# docker build -t iris-classifier .
# aws ecr get-login-password | docker login --username AWS --password-stdin <account>.dkr.ecr.<region>.amazonaws.com
# docker tag iris-classifier:latest <account>.dkr.ecr.<region>.amazonaws.com/iris-classifier:latest
# docker push <account>.dkr.ecr.<region>.amazonaws.com/iris-classifier:latest

print('Lambda container Dockerfile:\n', DOCKERFILE_LAMBDA)

Lambda container Dockerfile:
 
FROM public.ecr.aws/lambda/python:3.11

# Install dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy model and handler
COPY model.joblib ./model.joblib
COPY lambda_handler.py ./lambda_handler.py

# Lambda entry point
CMD ["lambda_handler.lambda_handler"]



## AWS SageMaker Endpoints

```python
import boto3
import sagemaker
from sagemaker.sklearn import SKLearnModel
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

session = sagemaker.Session()
role = sagemaker.get_execution_role()

# Upload model artifact
model_path = session.upload_data('model.tar.gz', bucket=session.default_bucket(), key_prefix='iris-model')

# Create SKLearn model
model = SKLearnModel(
    model_data=model_path,
    role=role,
    entry_point='inference.py',
    framework_version='1.2-1',
    py_version='py3'
)

# Real-time endpoint
predictor = model.deploy(
    initial_instance_count=1,
    instance_type='ml.t2.medium',
    endpoint_name='iris-classifier-v1'
)
predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()

# Inference
result = predictor.predict({'features': [5.1, 3.5, 1.4, 0.2]})
print(result)

# Async endpoint (for large payloads)
async_predictor = model.deploy(
    instance_type='ml.m5.xlarge',
    async_inference_config=sagemaker.async_inference.AsyncInferenceConfig(
        output_path='s3://bucket/output/'
    )
)

# Batch Transform (offline scoring)
transformer = model.transformer(
    instance_count=2,
    instance_type='ml.m5.xlarge',
    output_path='s3://bucket/batch-output/'
)
transformer.transform('s3://bucket/batch-input/', content_type='text/csv')
transformer.wait()
```

## Modal ML-Native Serverless

```python
# pip install modal
import modal

app = modal.App("iris-classifier")

# Define image with dependencies
image = modal.Image.debian_slim().pip_install("scikit-learn", "numpy")

@app.function(
    image=image,
    gpu="T4",           # optional GPU
    memory=2048,
    timeout=120,
    scaledown_window=300  # keep warm for 5 min
)
def predict(features: list) -> dict:
    import numpy as np
    import joblib
    model = joblib.load('/model/iris.joblib')  # loaded from Modal volume
    X = np.array([features])
    pred = model.predict(X)[0]
    proba = model.predict_proba(X)[0].tolist()
    return {'prediction': int(pred), 'probabilities': proba}

@app.local_entrypoint()
def main():
    result = predict.remote([5.1, 3.5, 1.4, 0.2])
    print(result)

# modal deploy app.py
# modal run app.py
```

## GCP Cloud Run

```dockerfile
# Dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8080"]
```

```bash
# Build and deploy
gcloud builds submit --tag gcr.io/PROJECT_ID/iris-classifier
gcloud run deploy iris-classifier \
  --image gcr.io/PROJECT_ID/iris-classifier \
  --platform managed \
  --region us-central1 \
  --allow-unauthenticated \
  --memory 2Gi \
  --cpu 2 \
  --max-instances 10
```

## Additional Learning Resources

### AWS
- [AWS Lambda Docs](https://docs.aws.amazon.com/lambda/)
- [SageMaker Docs](https://docs.aws.amazon.com/sagemaker/)
- [Serverless ML on AWS](https://aws.amazon.com/blogs/machine-learning/)

### Modal
- [Modal Docs](https://modal.com/docs)
- [Modal Examples](https://modal.com/docs/examples)

### GCP
- [Cloud Run Docs](https://cloud.google.com/run/docs)
- [Vertex AI Docs](https://cloud.google.com/vertex-ai/docs)

### Courses
- [AWS ML Specialty Certification](https://aws.amazon.com/certification/certified-machine-learning-specialty/)
- [MLOps Zoomcamp Deployment module](https://github.com/DataTalksClub/mlops-zoomcamp)